# Data Source Investigation

Explore source sample artifacts produced by `python ingestion/minimal_ingest.py` and export summary tables to CSV.


In [25]:
from pathlib import Path
import json
import zipfile

import pandas as pd


In [26]:
SAMPLES_DIR = Path('../../ingestion/tmp/source_samples')
REPORT_PATH = Path('../../ingestion/tmp/minimal_samples/minimal_ingestion_report.json')
EXPORT_DIR = Path('../../exploration/tmp/notebook_exports/01_data_source_investigation')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if not SAMPLES_DIR.exists():
    raise FileNotFoundError(f'Missing samples dir: {SAMPLES_DIR.resolve()}')

all_files = sorted([p for p in SAMPLES_DIR.rglob('*') if p.is_file()])
print('Samples dir:', SAMPLES_DIR.resolve())
print('Sample files:', len(all_files))
print('Export dir:', EXPORT_DIR.resolve())


Samples dir: /Users/ColeHoffman/dev/gis_phl/gis-phl/ingestion/tmp/source_samples
Sample files: 14
Export dir: /Users/ColeHoffman/dev/gis_phl/gis-phl/exploration/tmp/notebook_exports/01_data_source_investigation


In [27]:
rows = []
for p in all_files:
    rows.append({
        'source': p.parent.name,
        'file': p.name,
        'path': str(p),
        'bytes': p.stat().st_size,
    })
summary_df = pd.DataFrame(rows).sort_values(['source', 'file']).reset_index(drop=True)
summary_df.to_csv(EXPORT_DIR / 'sample_file_inventory.csv', index=False)
summary_df


,source,file,path,bytes
0,fred,philly_unemployment_head.csv,../../ingestion/tmp/source_samples/fred/philly...,6493
1,ntad_amtrak,amtrak_sample.geojson,../../ingestion/tmp/source_samples/ntad_amtrak...,3789
2,opendataphilly_li_property_history,dataset_page.html,../../ingestion/tmp/source_samples/opendataphi...,10599
3,opendataphilly_rental_suitability,dataset_page.html,../../ingestion/tmp/source_samples/opendataphi...,14308
4,opendataphilly_rental_suitability,resource_head.bin,../../ingestion/tmp/source_samples/opendataphi...,250000
5,phl_property_bulk,property_asset_head.bin,../../ingestion/tmp/source_samples/phl_propert...,92
6,phl_property_bulk,property_download_page.html,../../ingestion/tmp/source_samples/phl_propert...,7105
7,septa_gtfs,gtfs_nested.zip,../../ingestion/tmp/source_samples/septa_gtfs/...,28826336
8,septa_gtfs,gtfs_public.zip,../../ingestion/tmp/source_samples/septa_gtfs/...,29444199
9,septa_gtfs,gtfs_public_head.zip,../../ingestion/tmp/source_samples/septa_gtfs/...,200000


In [28]:
if REPORT_PATH.exists():
    report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
    report_df = pd.DataFrame([{
        'name': r.get('name'),
        'ok': r.get('ok'),
        'error': r.get('error')
    } for r in report.get('results', [])])
    report_df.to_csv(EXPORT_DIR / 'probe_status.csv', index=False)
    report_df
else:
    print('No report found at', REPORT_PATH)


## Sample Preview Helpers


In [29]:
def preview_text(path: Path, n_chars: int = 1200):
    text = path.read_text(encoding='utf-8', errors='replace')
    print(text[:n_chars])

def preview_csv(path: Path, n_rows: int = 5):
    return pd.read_csv(path).head(n_rows)

def preview_zip(path: Path):
    try:
        with zipfile.ZipFile(path, 'r') as zf:
            names = zf.namelist()
        return pd.DataFrame({'zip_member': names})
    except zipfile.BadZipFile:
        raw = path.read_bytes()
        text_head = raw[:240].decode('utf-8', errors='replace').replace('\n', ' ')
        return pd.DataFrame([
            {
                'zip_member': '<not_a_zip>',
                'bytes': len(raw),
                'first_16_hex': raw[:16].hex(),
                'text_head': text_head,
            }
        ])

def preview_geojson(path: Path, n_features: int = 3):
    payload = json.loads(path.read_text(encoding='utf-8'))
    feats = payload.get('features', [])
    rows = []
    for f in feats[:n_features]:
        props = f.get('properties', {})
        rows.append({
            'Code': props.get('Code'),
            'StationName': props.get('StationName'),
            'City': props.get('City'),
            'State': props.get('State'),
            'StnType': props.get('StnType'),
        })
    return pd.DataFrame(rows)


## Source-by-Source Sample Exploration


In [30]:
fred_preview = preview_csv(SAMPLES_DIR / 'fred' / 'philly_unemployment_head.csv')
fred_preview.to_csv(EXPORT_DIR / 'fred_preview.csv', index=False)
fred_preview


,observation_date,PHIL942UR
0,1990-01-01,4.6
1,1990-02-01,4.6
2,1990-03-01,4.6
3,1990-04-01,4.6
4,1990-05-01,4.7


In [31]:
zillow_preview = preview_csv(SAMPLES_DIR / 'zillow' / 'zori_head.csv')
zillow_preview.to_csv(EXPORT_DIR / 'zillow_preview.csv', index=False)
zillow_preview


,RegionID,SizeRank,RegionName,RegionType,StateName,2015-01-31,2015-02-28,2015-03-31,2015-04-30,2015-05-31,...,2025-04-30,2025-05-31,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31
0,102001,0,United States,country,NaN,1136.875660,1143.091571,1151.628492,1160.334639,1168.937100,...,1891.718540,1900.093803,1905.555874,1907.970919,1908.425577,1906.815052,1903.136022,1897.457476,1893.079810,1894.812508
1,394913,1,"New York, NY",msa,NY,2147.623068,2162.038487,2180.440072,2198.666017,2212.859042,...,3175.002413,3202.184453,3232.903750,3261.988647,3282.402140,3279.629635,3266.917419,3246.041279,3234.192899,3232.316444
2,753899,2,"Los Angeles, CA",msa,CA,1745.689589,1756.971610,1772.234004,1787.247740,1801.898197,...,2887.746640,2892.553080,2897.174510,2900.108469,2901.111619,2900.674280,2897.770375,2890.957145,2881.996091,2884.813189
3,394463,3,"Chicago, IL",msa,IL,1343.902521,1350.949124,1360.799161,1369.910884,1379.609447,...,2042.403060,2061.841565,2079.868003,2090.749547,2094.633714,2092.993114,2087.380071,2082.309385,2080.267683,2091.226250
4,394514,4,"Dallas, TX",msa,TX,1045.936211,1050.668328,1058.120800,1069.218528,1078.410916,...,1659.405853,1663.923737,1665.448891,1662.221097,1658.261579,1653.220900,1646.828823,1640.644284,1635.001553,1632.977977


In [32]:
amtrak_preview = preview_geojson(SAMPLES_DIR / 'ntad_amtrak' / 'amtrak_sample.geojson')
amtrak_preview.to_csv(EXPORT_DIR / 'ntad_amtrak_preview.csv', index=False)
amtrak_preview


,Code,StationName,City,State,StnType
0,AAM,"Alma, MI",Alma,MI,BUS
1,ABA,"Albany, NY",Albany,NY,BUS
2,ABB,"Abbotsford-Colby, WI",Colby,WI,BUS


In [33]:
gtfs_zip_path = SAMPLES_DIR / 'septa_gtfs' / 'gtfs_public.zip'
if not gtfs_zip_path.exists():
    gtfs_zip_path = SAMPLES_DIR / 'septa_gtfs' / 'gtfs_public_head.zip'
septa_zip_members = preview_zip(gtfs_zip_path)
septa_zip_members.to_csv(EXPORT_DIR / 'septa_gtfs_zip_members.csv', index=False)
septa_zip_members


,zip_member
0,google_bus.zip
1,google_rail.zip


In [21]:
preview_text(SAMPLES_DIR / 'septa_gtfs_rt' / 'gtfs_rt_metadata_sample.html', n_chars=1000)


<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html xmlns="http://www.w3.org/1999/xhtml">
<head>

<meta http-equiv="Content-Type" content="text/html; charset=utf-8" />
<meta http-equiv="X-UA-Compatible" content="IE=EmulateIE7" />
<title>SEPTA | GTFS Developer Download</title>
<link rel="icon" type="image/png" href="/main/images/septawhite.png" />      
<link rel="stylesheet" href="/main/css/reset.css" media="all" />
<link rel="stylesheet" href="/main/css/main.css" media="all" />
<link rel="stylesheet" href="/main/css/menu_style.css" media="all"/>
<link type="text/css" href="/main/css/ui.all.css" rel="stylesheet" />
<link type="text/css" href="/main/css/ui.datepicker.css" rel="stylesheet" />
<link type="text/css" href="/main/css/ui.theme.css" rel="stylesheet" />

<script language="JavaScript" src="/main/js/datefinder.js"></script>
<script type="text/javascript">var RecaptchaOptions = {theme : 'clean'};</script>


In [22]:
preview_text(SAMPLES_DIR / 'phl_property_bulk' / 'property_download_page.html', n_chars=1000)


<html>
  <head>
    <meta charset="utf-8">
    <meta http-equiv="X-UA-Compatible" content="IE=edge">
    <meta name="viewport" content="width=device-width, initial-scale=1">

    <title>Property | phila.gov</title>
    <link rel='icon' type='image/x-icon' href="//cityofphiladelphia.github.io/patterns/images/favicon.ico">
    <meta name="description" content="">

    <link rel="stylesheet" href="//maxcdn.bootstrapcdn.com/font-awesome/4.3.0/css/font-awesome.min.css">
    <!--Ionicons are optional-->
    <link rel="stylesheet" href="//code.ionicframework.com/ionicons/2.0.1/css/ionicons.min.css">

    <!--pattern stylesheet includes foundation css -->
    <link rel="stylesheet" href="//cityofphiladelphia.github.io/patterns/dist/0.9.1/css/patterns.css">

    <link rel="stylesheet" href="styles.css">

    <link rel="canonical" href="">

    <script src="//cdnjs.cloudflare.com/ajax/libs/modernizr/2.8.3/modernizr.js"></script>
  </head>

  <body>
    <!-- Google Tag Manager [phila.gov] -->
   

In [23]:
preview_text(SAMPLES_DIR / 'opendataphilly_rental_suitability' / 'dataset_page.html', n_chars=1000)


<!DOCTYPE html>
<html class="no-js" prefix="dct: http://purl.org/dc/terms/
              rdf: http://www.w3.org/1999/02/22-rdf-syntax-ns#
              dcat: http://www.w3.org/ns/dcat#
              foaf: http://xmlns.com/foaf/0.1/">

  <head>
  <meta charset="utf-8">
  <meta http-equiv="X-UA-Compatible" content="IE=edge">
  <meta name="viewport" content="width=device-width, initial-scale=1">

  <title>Certified for Rental Suitability - OpenDataPhilly</title>
  <meta name="description" content="">

  <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.0.2/dist/css/bootstrap.min.css" rel="stylesheet" integrity="sha384-EVSTQN3/azprG1Anm3QDgpJLIm9Nao0Yz1ztcQTwFspd3yD65VohhpuuCOmLASjC" crossorigin="anonymous">
  <link rel="shortcut icon" type="image/png" href="/img/favicon.png" >
  <link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Lato">
  <link rel="stylesheet" href="https://maxcdn.bootstrapcdn.com/font-awesome/4.5.0/css/font-awesome.min.css">
  <link rel="stylesheet"

In [24]:
preview_text(SAMPLES_DIR / 'opendataphilly_li_property_history' / 'dataset_page.html', n_chars=1000)


<!DOCTYPE html>
<html class="no-js" prefix="dct: http://purl.org/dc/terms/
              rdf: http://www.w3.org/1999/02/22-rdf-syntax-ns#
              dcat: http://www.w3.org/ns/dcat#
              foaf: http://xmlns.com/foaf/0.1/">

  <head>
  <meta charset="utf-8">
  <meta http-equiv="X-UA-Compatible" content="IE=edge">
  <meta name="viewport" content="width=device-width, initial-scale=1">

  <title>Licenses and Inspections Property History - OpenDataPhilly</title>
  <meta name="description" content="">

  <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.0.2/dist/css/bootstrap.min.css" rel="stylesheet" integrity="sha384-EVSTQN3/azprG1Anm3QDgpJLIm9Nao0Yz1ztcQTwFspd3yD65VohhpuuCOmLASjC" crossorigin="anonymous">
  <link rel="shortcut icon" type="image/png" href="/img/favicon.png" >
  <link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Lato">
  <link rel="stylesheet" href="https://maxcdn.bootstrapcdn.com/font-awesome/4.5.0/css/font-awesome.min.css">
  <link rel="st